# Pruebas estadísticas, modelos y clasificación

Cuaderno del módulo *Periodismo de Precisión*. Reúne ejemplos cortos de las técnicas de la sección **¿La diferencia es real?** y de las dos que siguen: **¿Qué explica Y?** y **Clasificación**. La idea no es dominarlas todas, sino ver cómo se ven y cómo se leen sus resultados, para profundizar en las que le interesen.

Usamos la base del ejercicio 2: **clima y violencia en Barranquilla, 2010-2016** (Trujillo y Howley, 2018). Cada fila es un día.

Ejecute las celdas en orden, con el botón ▶ o con `Shift + Enter`.

## 0. Preparar los datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.contingency_tables import mcnemar

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

URL = "https://raw.githubusercontent.com/nescolar/periodismo-de-precision/main/assets/datos/Dataset_clima_violencia_barranquilla.xlsx"
df = pd.read_excel(URL, sheet_name="Datos")

# Variables nuevas para los ejemplos
df["llovio"] = np.where(df["lluvia"] > 0, "Sí", "No")
df["domingo"] = (df["dia_semana"] == "Domingo").astype(int)
# Un día "muy violento" es un día atípico: más de 21 casos (regla de 1,5 × RIC, ejercicio 2)
df["dia_muy_violento"] = (df["violencia_interpersonal"] > 21).astype(int)

print(df.shape[0], "días;", df["dia_muy_violento"].sum(), "días muy violentos")

Esta función nos ayuda a leer el valor p de todas las pruebas con el mismo criterio: si p es menor que 0,05, se rechaza la hipótesis nula (H₀).

In [ ]:
def decidir(p, alfa=0.05):
    if p < alfa:
        print(f"p = {p:.4f} < {alfa}: se rechaza H₀. La diferencia o relación es estadísticamente significativa.")
    else:
        print(f"p = {p:.4f} ≥ {alfa}: no se rechaza H₀. No hay evidencia suficiente de diferencia o relación.")

## 1. Pruebas de comparación

### 1.1 Diferencia de medias: prueba t de Student

**Pregunta:** ¿los días con lluvia tienen, en promedio, un número distinto de casos de violencia que los días sin lluvia?

- H₀: el promedio de casos es igual con lluvia y sin lluvia.
- H₁: es distinto.

Usamos la versión de Welch (`equal_var=False`), que no supone que los dos grupos varíen lo mismo.

In [ ]:
print(df.groupby("llovio")["violencia_interpersonal"].agg(Días="count", Promedio="mean"))

con = df.loc[df["llovio"] == "Sí", "violencia_interpersonal"]
sin = df.loc[df["llovio"] == "No", "violencia_interpersonal"]
t, p = stats.ttest_ind(con, sin, equal_var=False)
print(f"\nDiferencia: {con.mean() - sin.mean():.2f} casos por día")
decidir(p)

**Para comentar:** la diferencia es significativa, pero ¿es grande? ¿Cómo la contaría en una nota?

### 1.2 Diferencia de medias antes y después: prueba t pareada

Se usa cuando se mide **a las mismas personas dos veces**, como en la [nota de ejemplo](https://nescolar.github.io/periodismo-de-precision/nota/). Esta base no tiene ese tipo de datos, así que usamos **datos ilustrativos**: el puntaje de 8 estudiantes en una prueba de 15 preguntas, antes y después de un taller.

- H₀: el promedio no cambió.
- H₁: cambió.

In [ ]:
antes   = np.array([9, 11, 10, 12, 8, 13, 10, 11])
despues = np.array([11, 12, 12, 13, 9, 14, 12, 11])

print(f"Promedio antes: {antes.mean():.2f} | después: {despues.mean():.2f}")
t, p = stats.ttest_rel(despues, antes)
decidir(p)

### 1.3 Diferencia de proporciones: prueba z de dos proporciones

**Pregunta:** ¿el porcentaje de días muy violentos es distinto los fines de semana que entre semana?

- H₀: el porcentaje es igual en los dos grupos.
- H₁: es distinto.

In [ ]:
fin = df[df["fin_de_semana"] == 1]
semana = df[df["fin_de_semana"] == 0]

casos = [fin["dia_muy_violento"].sum(), semana["dia_muy_violento"].sum()]
dias = [len(fin), len(semana)]
print(f"Fin de semana: {casos[0]} de {dias[0]} días ({casos[0] / dias[0]:.1%})")
print(f"Entre semana:  {casos[1]} de {dias[1]} días ({casos[1] / dias[1]:.1%})")

z, p = proportions_ztest(casos, dias)
decidir(p)

### 1.4 Diferencia de proporciones antes y después: prueba de McNemar

También para las **mismas personas medidas dos veces**, pero con una respuesta de sí o no. Usamos **datos ilustrativos**: 55 estudiantes respondieron antes y después si confían en estudiantes de otros colegios.

La prueba se fija en quienes **cambiaron** de respuesta: los que pasaron de no a sí y los que pasaron de sí a no.

- H₀: el porcentaje que responde sí no cambió.
- H₁: cambió.

In [ ]:
#                 Después: No   Después: Sí
tabla = [[28, 15],   # Antes: No
         [3,  9]]    # Antes: Sí

print("Antes respondían sí:", 3 + 9, "de 55 |", "Después:", 15 + 9, "de 55")
resultado = mcnemar(tabla, exact=True)
decidir(resultado.pvalue)

### 1.5 Tres o más grupos: ANOVA

**Pregunta:** ¿el promedio de casos cambia según el día de la semana?

- H₀: el promedio es igual los siete días.
- H₁: al menos un día es distinto.

In [ ]:
orden = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
grupos = [df.loc[df["dia_semana"] == d, "violencia_interpersonal"] for d in orden]

print(df.groupby("dia_semana")["violencia_interpersonal"].mean().reindex(orden).round(1))
F, p = stats.f_oneway(*grupos)
decidir(p)

# La alternativa que compara el orden de los datos (útil si no tienen forma de campana)
H, p_kw = stats.kruskal(*grupos)
print(f"Kruskal-Wallis: p = {p_kw:.4f}")

plt.figure(figsize=(8, 3.5))
plt.boxplot(grupos)
plt.xticks(range(1, 8), [d[:3] for d in orden])
plt.ylabel("Casos por día")
plt.title("Casos de violencia según el día de la semana", loc="left")
plt.show()

**Para comentar:** el ANOVA dice que al menos un día es distinto. Mirando el gráfico, ¿cuál cree que es?

### 1.6 Dos variables cualitativas: chi-cuadrado

**Pregunta:** ¿ser un día muy violento está asociado con el tipo de día?

- H₀: las dos variables son independientes: no hay asociación.
- H₁: están asociadas.

In [ ]:
tabla = pd.crosstab(df["tipo_dia"], df["dia_muy_violento"])
tabla.columns = ["No", "Sí"]
print(tabla)
print("\nPorcentaje de días muy violentos en cada tipo de día:")
print((tabla["Sí"] / tabla.sum(axis=1) * 100).round(1))

chi2, p, gl, esperados = stats.chi2_contingency(tabla)
decidir(p)

## 2. Pruebas de asociación: Pearson y Spearman

**Pregunta:** ¿los días más calientes tienen más casos de violencia?

- H₀: no hay relación (r = 0).
- H₁: sí la hay.

In [ ]:
r, p = stats.pearsonr(df["temp_max"], df["violencia_interpersonal"])
print(f"Pearson: r = {r:.3f}")
decidir(p)

rho, p_s = stats.spearmanr(df["temp_max"], df["violencia_interpersonal"])
print(f"\nSpearman: rho = {rho:.3f}, p = {p_s:.4f}")

**Para comentar:** con 2.557 días, una correlación de 0,04 sale **significativa**. ¿Significa eso que el calor importa? Es el mejor ejemplo de que **significativo no es lo mismo que importante**: la relación existe en los datos, pero es tan débil que no sirve para explicar la violencia.

## 3. ¿Qué explica Y? Regresión lineal

La regresión estima cuánto cambia la variable que nos interesa (**Y**, aquí los casos de violencia) cuando cambia cada variable explicativa (**X**), **manteniendo las demás constantes**. Por eso sirve para detectar terceras variables.

Vamos a ver qué pasa con los comparendos por embriaguez cuando agregamos el tipo de día al modelo.

In [ ]:
m1 = smf.ols("violencia_interpersonal ~ comparendos_embriaguez", data=df).fit()
m2 = smf.ols("violencia_interpersonal ~ comparendos_embriaguez + C(tipo_dia)", data=df).fit()
m3 = smf.ols("violencia_interpersonal ~ comparendos_embriaguez + C(tipo_dia) + temp_max + lluvia", data=df).fit()

def resumen(modelo, nombre):
    tabla = pd.DataFrame({"Coeficiente": modelo.params, "Valor p": modelo.pvalues})
    print(f"--- {nombre} | R² = {modelo.rsquared:.3f}")
    print(tabla.round(3), "\n")

resumen(m1, "Modelo 1: solo comparendos")
resumen(m2, "Modelo 2: comparendos + tipo de día")
resumen(m3, "Modelo 3: + temperatura y lluvia")

**Cómo se lee:**

- **Coeficiente:** cuánto cambia Y por cada unidad de X, con lo demás constante. En el modelo 1, cada comparendo adicional se asocia con unos 0,3 casos más de violencia. En el modelo 2, al tener en cuenta el tipo de día, ese coeficiente baja a la mitad (0,14): parte de la relación venía del calendario.
- **Coeficientes de una variable cualitativa** (`C(tipo_dia)`): se comparan con la categoría de referencia, que aquí es *Entre semana*. Un festivo tiene, en promedio, unos 5 casos más que un día entre semana con los mismos comparendos.
- **Valor p:** si es menor que 0,05, el coeficiente es significativamente distinto de cero.
- **R²:** qué parte de la variación de Y explica el modelo. Con 0,20, el modelo explica alrededor del 20 % de las diferencias entre días; el 80 % restante depende de cosas que no están en los datos.

**Para comentar:** en el modelo 3, la temperatura es significativa. ¿Cuántos casos más hay por cada grado de temperatura? ¿Cambia eso la respuesta a la pregunta guía del ejercicio 2?

### 3.2 Cuando Y es de sí o no: regresión logística

**Pregunta:** ¿qué tanto cambia la probabilidad de un día muy violento según el tipo de día y la temperatura?

In [ ]:
logit = smf.logit("dia_muy_violento ~ C(tipo_dia) + temp_max", data=df).fit(disp=0)
print(pd.DataFrame({"Coeficiente": logit.params, "Valor p": logit.pvalues}).round(3))

# Probabilidad estimada para cada tipo de día, con la temperatura promedio
nuevos = pd.DataFrame({"tipo_dia": ["Entre semana", "Fin de semana", "Festivo"],
                       "temp_max": df["temp_max"].mean()})
nuevos["Probabilidad de un día muy violento"] = logit.predict(nuevos)
print()
print(nuevos[["tipo_dia", "Probabilidad de un día muy violento"]])

Los coeficientes de la regresión logística son difíciles de leer. Para contarlos, conviene traducirlos a **probabilidades**, como en la tabla de arriba: con la misma temperatura, un festivo tiene muchas más probabilidades de ser un día muy violento que un día entre semana. La temperatura no es significativa.

## 4. Clasificación: ¿en qué grupo cae cada caso?

Un modelo de clasificación aprende de datos pasados a asignar cada caso a una categoría. Aquí, un **árbol de decisión** intenta adivinar si un día será muy violento a partir de su tipo, la temperatura, la lluvia y los comparendos.

Para saber si funciona, lo **entrenamos** con el 70 % de los días y lo **probamos** con el 30 % restante, que el modelo no ha visto.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

X = df[["festivo", "fin_de_semana", "domingo", "num_mes", "temp_max", "lluvia", "comparendos_embriaguez"]]
y = df["dia_muy_violento"]
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.3, random_state=2026, stratify=y)

print("Días de prueba:", len(y_pru), "| muy violentos:", y_pru.sum())
print(f"Si el modelo dijera siempre 'no', acertaría el {1 - y_pru.mean():.1%} de las veces")

In [ ]:
def evaluar(modelo, nombre):
    pred = modelo.predict(X_pru)
    vp = ((pred == 1) & (y_pru == 1)).sum()
    print(f"--- {nombre}")
    print(f"Exactitud (aciertos sobre el total): {accuracy_score(y_pru, pred):.1%}")
    print(f"Días muy violentos detectados: {vp} de {y_pru.sum()} ({recall_score(y_pru, pred):.0%})")
    print(f"Falsas alarmas: {((pred == 1) & (y_pru == 0)).sum()}")
    print(pd.DataFrame(confusion_matrix(y_pru, pred),
                       index=["Real: no", "Real: sí"], columns=["Predicho: no", "Predicho: sí"]), "\n")

arbol = DecisionTreeClassifier(max_depth=3, random_state=2026).fit(X_ent, y_ent)
evaluar(arbol, "Árbol sin ajustar")

arbol_b = DecisionTreeClassifier(max_depth=3, random_state=2026, class_weight="balanced").fit(X_ent, y_ent)
evaluar(arbol_b, "Árbol que da más peso a los días muy violentos")

**La trampa de la exactitud.** Como solo el 4 % de los días son muy violentos, un modelo que dice siempre "no" acierta el 96 % de las veces sin servir para nada. El primer árbol tiene una exactitud parecida, pero detecta muy pocos días muy violentos. El segundo, que da más peso a la categoría rara, acierta menos en total, pero detecta la mayoría, a cambio de muchas falsas alarmas.

**La matriz de confusión** muestra los cuatro resultados posibles: aciertos en cada categoría, falsas alarmas (predijo sí y era no) y casos perdidos (predijo no y era sí). Elegir entre falsas alarmas y casos perdidos es una decisión que depende del uso del modelo.

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(arbol_b, feature_names=X.columns, class_names=["No", "Sí"], filled=True, rounded=True, fontsize=8)
plt.show()

**Cómo se lee el árbol:** cada caja hace una pregunta sobre una variable (por ejemplo, `comparendos_embriaguez <= 2.5`). Si la respuesta es sí, se sigue por la izquierda; si es no, por la derecha. Al final de cada camino está la categoría que el modelo predice.

**Para comentar:** ¿qué variables usa el árbol para decidir? ¿Coinciden con lo que encontró en el ejercicio 2?

**Un cuidado ético.** Este modelo clasifica días. Cuando un modelo clasifica **personas** (para dar un crédito, priorizar una inspección o predecir un delito), sus errores y sesgos afectan a gente real. Antes de reportar sobre un algoritmo, pregunte con qué datos se entrenó, cuántas falsas alarmas y cuántos casos perdidos tiene, y a quién afectan.